# Предварительный анализ сырых данных

Первичный осмотр CSV-файлов соревнования Corporación Favorita Grocery Sales Forecasting из `data/landing`: типы столбцов, диапазоны, пропуски, дубли и проверки ключей. Запросы выполняются в DuckDB напрямую по CSV, вспомогательные функции профилирования — в `scripts/duckdb_functions.py`.

Выводы сведены в `docs_favorita/0_raw_data_exploration/data_dictionary.md`.

## Подготовка

Импорты.

In [52]:
import duckdb
from scripts import duckdb_functions as dbf

Подключение к DuckDB и ограничение памяти, чтобы обработка train (~125 млн строк) не вытесняла Windows и Jupyter.

In [53]:
con = duckdb.connect("data/favorita.duckdb")
con.sql("SET memory_limit = '8GB'")            # оставить память Windows и Jupyter
con.sql("SET temp_directory = 'data/tmp'")     # куда сбрасывать при нехватке RAM
con.sql("SET preserve_insertion_order = false"); # меньше памяти при больших COPY

Веб-интерфейс DuckDB для ручного просмотра, при необходимости.

In [54]:
# con.sql("CALL start_ui()");

Пути к исходным файлам.

In [55]:
train_table_path = "'data/landing/train.csv'"
stores_table_path = "'data/landing/stores.csv'"
oil_table_path = "'data/landing/oil.csv'"
holidays_table_path = "'data/landing/holidays_events.csv'"
items_table_path = "'data/landing/items.csv'"
transaction_table_path = "'data/landing/transactions.csv'"
test_table_path = "'data/landing/test.csv'"

## Train

История продаж: товар × магазин × день.

Столбцы и типы, которые DuckDB определил при чтении CSV.

In [56]:
dbf.show_columns(con, train_table_path)

,dtype,kind
id,BIGINT,int
date,DATE,date
store_nbr,BIGINT,int
item_nbr,BIGINT,int
unit_sales,DOUBLE,float
onpromotion,VARCHAR,cat


`id` — идентификатор строки.

In [57]:
dbf.show_num_stats(con, train_table_path, 'id')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
id,id,BIGINT,125497040,125497040,0,0,0,125497039,False,True,0


`date` — дата продаж. Проверяем диапазон и пропущенные дни.

In [58]:
dbf.show_date_stats(con, train_table_path, 'date')

,column,dtype,total,not_null,nulls,duplicates,min,max,span_days,distinct_days,continuous,missing_in_range,gaps,first_gap_start,first_gap_end
date,date,DATE,125497040,125497040,0,125495356,2013-01-01,2017-08-15,1688,1684,False,4,4,2013-12-25,2013-12-25


`store_nbr` — номер магазина.

In [59]:
dbf.show_num_stats(con, train_table_path, 'store_nbr')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
store_nbr,store_nbr,BIGINT,125497040,125497040,0,125496986,1,54,False,True,0


`item_nbr` — номер товара.

In [60]:
dbf.show_num_stats(con, train_table_path, 'item_nbr')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
item_nbr,item_nbr,BIGINT,125497040,125497040,0,125493004,96995,2127114,False,False,2026084


`unit_sales` — объём продаж, целевая переменная.

In [61]:
dbf.show_num_stats(con, train_table_path, 'unit_sales')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
unit_sales,unit_sales,DOUBLE,125497040,125497040,0,125238566,-15372.0,89440.0,True,None,None


`onpromotion` — признак акции. DuckDB прочитал его как строку, поэтому смотрим как категориальный.

In [62]:
dbf.show_cat_stats(con, train_table_path, 'onpromotion')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
onpromotion,onpromotion,VARCHAR,125497040,103839389,21657651,103839387,2,0.0


### Проверки ключей

Комбинация (`date`, `store_nbr`, `item_nbr`) уникальна.

In [63]:
assert len(con.execute(f"select date, store_nbr, item_nbr, count(*) as count \
            from {train_table_path}\
            group by date, store_nbr, item_nbr\
            having count(*)>1").fetch_df()) == 0

Каждый `store_nbr` из train есть в stores (внешний ключ).

In [64]:
assert con.execute(f"""
    select count(*)
    from {train_table_path} as train
    where not exists (
        select 1
        from {stores_table_path} as stores
        where stores.store_nbr = train.store_nbr)
""").fetchone()[0] == 0 # type: ignore

## Stores

Справочник магазинов.

Столбцы и типы.

In [65]:
dbf.show_columns(con, stores_table_path)

,dtype,kind
store_nbr,BIGINT,int
city,VARCHAR,cat
state,VARCHAR,cat
type,VARCHAR,cat
cluster,BIGINT,int


`store_nbr` — номер магазина.

In [66]:
dbf.show_num_stats(con, stores_table_path, 'store_nbr')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
store_nbr,store_nbr,BIGINT,54,54,0,0,1,54,False,True,0


`city` — город.

In [67]:
dbf.show_cat_stats(con, stores_table_path, 'city')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
city,city,VARCHAR,54,54,0,32,22,40.74


`state` — провинция.

In [68]:
dbf.show_cat_stats(con, stores_table_path, 'state')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
state,state,VARCHAR,54,54,0,38,16,29.63


`type` — тип магазина.

In [69]:
dbf.show_cat_stats(con, stores_table_path, 'type')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
type,type,VARCHAR,54,54,0,49,5,9.26


`cluster` — группа схожих магазинов.

In [70]:
dbf.show_num_stats(con, stores_table_path, 'cluster')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
cluster,cluster,BIGINT,54,54,0,37,1,17,False,True,0


`store_nbr` не содержит NULL и уникален, то есть может быть первичным ключом.

In [71]:
assert len(con.execute(f"select * \
            from {stores_table_path}\
            where store_nbr is null").fetch_df()) == 0

assert len(con.execute(f"select store_nbr, count(*) as count \
            from {stores_table_path}\
            group by  store_nbr\
            having count(*)>1").fetch_df()) == 0

## Oil

Дневная цена нефти.

Столбцы и типы.

In [72]:
dbf.show_columns(con, oil_table_path)

,dtype,kind
date,DATE,date
dcoilwtico,DOUBLE,float


`date` — дата котировки. Проверяем диапазон и пропущенные дни.

In [73]:
dbf.show_date_stats(con, oil_table_path, 'date')

,column,dtype,total,not_null,nulls,duplicates,min,max,span_days,distinct_days,continuous,missing_in_range,gaps,first_gap_start,first_gap_end
date,date,DATE,1218,1218,0,0,2013-01-01,2017-08-31,1704,1218,False,486,243,2013-01-05,2013-01-06


`dcoilwtico` — цена нефти WTI.

In [74]:
dbf.show_num_stats(con, oil_table_path, 'dcoilwtico')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
dcoilwtico,dcoilwtico,DOUBLE,1218,1175,43,177,26.19,110.62,True,None,None


`date` не содержит NULL и уникальна, то есть может быть первичным ключом.

In [75]:
assert len(con.execute(f"select * \
            from {oil_table_path}\
            where date is null").fetch_df()) == 0

assert len(con.execute(f"select date, count(*) as count \
            from {oil_table_path}\
            group by  date\
            having count(*)>1").fetch_df()) == 0

## Holidays_events

Праздники и события.

Столбцы и типы.

In [76]:
dbf.show_columns(con, holidays_table_path)

,dtype,kind
date,DATE,date
type,VARCHAR,cat
locale,VARCHAR,cat
locale_name,VARCHAR,cat
description,VARCHAR,cat
transferred,BOOLEAN,cat


`date` — дата праздника или события.

In [77]:
dbf.show_date_stats(con, holidays_table_path, 'date')

,column,dtype,total,not_null,nulls,duplicates,min,max,span_days,distinct_days,continuous,missing_in_range,gaps,first_gap_start,first_gap_end
date,date,DATE,350,350,0,38,2012-03-02,2017-12-26,2126,312,False,1814,178,2012-03-03,2012-03-31


`type` — тип дня.

In [78]:
dbf.show_cat_stats(con, holidays_table_path, 'type')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
type,type,VARCHAR,350,350,0,344,6,1.71


Значения `type`.

In [79]:
con.execute(f"select distinct type \
            from {holidays_table_path}").fetch_df().T

,0,1,2,3,4,5
type,Work Day,Event,Transfer,Bridge,Additional,Holiday


`locale` — уровень действия праздника.

In [80]:
dbf.show_cat_stats(con, holidays_table_path, 'locale')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
locale,locale,VARCHAR,350,350,0,347,3,0.86


Значения `locale`.

In [81]:
con.execute(f"select distinct locale \
            from {holidays_table_path}").fetch_df().T

,0,1,2
locale,Local,National,Regional


`locale_name` — территория действия.

In [82]:
dbf.show_cat_stats(con, holidays_table_path, 'locale_name')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
locale_name,locale_name,VARCHAR,350,350,0,326,24,6.86


`description` — название праздника.

In [83]:
dbf.show_cat_stats(con, holidays_table_path, 'description')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
description,description,VARCHAR,350,350,0,247,103,29.43


`transferred` — признак переноса праздника.

In [84]:
dbf.show_cat_stats(con, holidays_table_path, 'transferred')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
transferred,transferred,BOOLEAN,350,350,0,348,2,0.57


## Items

Справочник товаров.

Столбцы и типы.

In [85]:
dbf.show_columns(con, items_table_path)

,dtype,kind
item_nbr,BIGINT,int
family,VARCHAR,cat
class,BIGINT,int
perishable,BIGINT,int


`item_nbr` — номер товара.

In [86]:
dbf.show_num_stats(con, items_table_path, 'item_nbr')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
item_nbr,item_nbr,BIGINT,4100,4100,0,0,96995,2134244,False,False,2033150


`family` — товарное семейство.

In [87]:
dbf.show_cat_stats(con, items_table_path, 'family')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
family,family,VARCHAR,4100,4100,0,4067,33,0.8


`class` — код товарного класса.

In [88]:
dbf.show_num_stats(con, items_table_path, 'class')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
class,class,BIGINT,4100,4100,0,3763,1002,7780,False,False,6442


`perishable` — признак скоропортящегося товара.

In [89]:
dbf.show_num_stats(con, items_table_path, 'perishable')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
perishable,perishable,BIGINT,4100,4100,0,4098,0,1,False,True,0


`item_nbr` не содержит NULL и уникален, то есть может быть первичным ключом.

In [90]:
assert len(con.execute(f"select * \
            from {items_table_path}\
            where item_nbr is null").fetch_df()) == 0

assert len(con.execute(f"select item_nbr, count(*) as count \
            from {items_table_path}\
            group by  item_nbr\
            having count(*)>1").fetch_df()) == 0

## Transactions

Число чеков по магазинам и дням.

Столбцы и типы.

In [91]:
dbf.show_columns(con, transaction_table_path)

,dtype,kind
date,DATE,date
store_nbr,BIGINT,int
transactions,BIGINT,int


`date` — дата. Проверяем диапазон и пропущенные дни.

In [92]:
dbf.show_date_stats(con, transaction_table_path, 'date')

,column,dtype,total,not_null,nulls,duplicates,min,max,span_days,distinct_days,continuous,missing_in_range,gaps,first_gap_start,first_gap_end
date,date,DATE,83488,83488,0,81806,2013-01-01,2017-08-15,1688,1682,False,6,6,2013-12-25,2013-12-25


`store_nbr` — номер магазина.

In [93]:
dbf.show_num_stats(con, transaction_table_path, 'store_nbr')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
store_nbr,store_nbr,BIGINT,83488,83488,0,83434,1,54,False,True,0


`transactions` — число чеков в магазине за день.

In [94]:
dbf.show_num_stats(con, transaction_table_path, 'transactions')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
transactions,transactions,BIGINT,83488,83488,0,78495,5,8359,False,False,3362


## Test

Пары товар × магазин × день, для которых нужен прогноз.

Столбцы и типы.

In [95]:
dbf.show_columns(con, test_table_path)

,dtype,kind
id,BIGINT,int
date,DATE,date
store_nbr,BIGINT,int
item_nbr,BIGINT,int
onpromotion,BOOLEAN,cat


`id` — идентификатор строки, продолжает нумерацию train.

In [96]:
dbf.show_num_stats(con, test_table_path, 'id')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
id,id,BIGINT,3370464,3370464,0,0,125497040,128867503,False,True,0


`date` — даты прогноза.

In [97]:
dbf.show_date_stats(con, test_table_path, 'date')

,column,dtype,total,not_null,nulls,duplicates,min,max,span_days,distinct_days,continuous,missing_in_range,gaps,first_gap_start,first_gap_end
date,date,DATE,3370464,3370464,0,3370448,2017-08-16,2017-08-31,16,16,True,0,0,None,None


`store_nbr` — номер магазина.

In [98]:
dbf.show_num_stats(con, test_table_path, 'store_nbr')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
store_nbr,store_nbr,BIGINT,3370464,3370464,0,3370410,1,54,False,True,0


`item_nbr` — номер товара.

In [99]:
dbf.show_num_stats(con, test_table_path, 'item_nbr')

,column,dtype,total,not_null,nulls,duplicates,min,max,has_fraction,continuous,missing_in_range
item_nbr,item_nbr,BIGINT,3370464,3370464,0,3366563,96995,2134244,False,False,2033349


`onpromotion` — признак акции.

In [100]:
dbf.show_cat_stats(con, test_table_path, 'onpromotion')

,column,dtype,total,not_null,nulls,duplicates,cardinality,cardinality_pct
onpromotion,onpromotion,BOOLEAN,3370464,3370464,0,3370462,2,0.0


### Проверки ключей

Комбинация (`date`, `store_nbr`, `item_nbr`) уникальна.

In [101]:
assert len(con.execute(f"select date, store_nbr, item_nbr, count(*) as count \
            from {test_table_path}\
            group by date, store_nbr, item_nbr\
            having count(*)>1").fetch_df()) == 0

Каждый `store_nbr` из test есть в stores (внешний ключ).

In [102]:
assert con.execute(f"""
    select count(*)
    from {test_table_path} as test
    where not exists (
        select 1
        from {stores_table_path} as stores
        where stores.store_nbr = test.store_nbr)
""").fetchone()[0] == 0 # type: ignore